In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/spacy-ner/cocktail_bio.csv
/kaggle/input/spacy-ner/dev.spacy
/kaggle/input/spacy-ner/config.cfg
/kaggle/input/spacy-ner/train.spacy


In [2]:
!pip install -U spacy[transformer,cuda12x]

INFO: pip is looking at multiple versions of thinc to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.3/82.3 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 81.6 MB/s eta 0:00:00
  Attempting uninstall: blis
    Found existing installation: blis 1.3.0
    Uninstalling blis-1.3.0:
      Successfully uninstalled blis-1.3.0
  Attempting uninstall: thinc
    Found existing installation: thinc 8.3.6
    Uninstalling thinc-8.3.6:
      Successfully uninstalled thinc-8.3.6
  Attempting uninstall: cupy-cuda12x
    Found existing installation: cupy-cuda12x 13.6.0
    Uninstalling cupy-cuda12x-13.6.0:
      Successfully uninstalled cupy-cuda12x-13.6.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following

In [3]:
import spacy
from spacy.tokens import DocBin
from spacy.util import minibatch
from pathlib import Path
import pandas as pd
import random
import shutil
import json
import torch

In [4]:
# ========= CONFIG =========
CSV_PATH = "/kaggle/input/ner-ingredient-bio/train.csv"
OUTPUT_DIR = Path("./output")
N_EPOCHS = 15
MODEL_NAME = "en_core_web_trf"  # Transformer model
GPU = 0 if torch.cuda.is_available() else -1
SEED = 42
VAL_SPLIT = 0.2

In [5]:
# ========= LOAD CSV =========
print("📥 Loading data...")
df = pd.read_csv("/kaggle/input/spacy-ner/cocktail_bio.csv")
df.drop(df.columns[0], axis=1, inplace=True)
required_cols = {"Sentence", "Labels"} # <-- CHANGED
assert required_cols.issubset(df.columns), f"CSV must contain {required_cols}"

# Create lists of tokens and labels from the strings
df['tokens'] = df['Sentence'].apply(lambda x: x.split())
df['labels'] = df['Labels'].apply(lambda x: x.split())

# Data validation: Ensure tokens and labels count match for each sentence
for i, row in df.iterrows():
    assert len(row['tokens']) == len(row['labels']), \
        f"Mismatch in row {i}: {len(row['tokens'])} tokens vs {len(row['labels'])} labels"

print("✅ Data loaded and tokenized.")

📥 Loading data...
✅ Data loaded and tokenized.


In [6]:
# Shuffle the entire DataFrame
df_shuffled = df.sample(frac=1, random_state=SEED)

# Split the shuffled DataFrame
split_idx = int(len(df_shuffled) * (1 - VAL_SPLIT))
train_df = df_shuffled.iloc[:split_idx]
val_df = df_shuffled.iloc[split_idx:]
print(f"✅ Train sentences: {len(train_df)} | Val sentences: {len(val_df)}")


✅ Train sentences: 8000 | Val sentences: 2000


In [7]:
# ========= BIO to ENT conversion =========
def bio_to_ents(tokens, labels):
    ents = []
    start, end, ent_label = None, None, None
    text = " ".join(tokens)
    char_idx = 0
    for i, (tok, lab) in enumerate(zip(tokens, labels)):
        tok_start = text.find(tok, char_idx)
        tok_end = tok_start + len(tok)
        char_idx = tok_end
        if lab.startswith("B-"):
            if ent_label is not None:
                ents.append((start, end, ent_label))
            start, end, ent_label = tok_start, tok_end, lab[2:]
        elif lab.startswith("I-") and ent_label == lab[2:]:
            end = tok_end
        else:
            if ent_label is not None:
                ents.append((start, end, ent_label))
                ent_label = None
    if ent_label is not None:
        ents.append((start, end, ent_label))
    return text, {"entities": ents}


In [8]:
train_data = [bio_to_ents(row.tokens, row.labels) for _, row in train_df.iterrows()]
val_data = [bio_to_ents(row.tokens, row.labels) for _, row in val_df.iterrows()]

In [9]:
# ========= INIT MODEL =========
if spacy.util.is_package(MODEL_NAME):
    nlp = spacy.load(MODEL_NAME)
else:
    nlp = spacy.blank("en")

if "ner" not in nlp.pipe_names:
    ner = nlp.add_pipe("ner", last=True)
else:
    ner = nlp.get_pipe("ner")

for _, annotations in train_data:
    for ent in annotations.get("entities"):
        ner.add_label(ent[2])

print("✅ Model initialized with labels:", ner.labels)

✅ Model initialized with labels: ('D/F', 'NAME', 'QUANTITY', 'STATE', 'TEMPERATURE', 'UNIT')


In [10]:
# ========= TRAINING =========
if GPU >= 0:
    spacy.require_gpu()
    print("🟢 Using GPU")

# ✅ Correct optimizer init for transformer models
optimizer = nlp.initialize(lambda: [spacy.training.Example.from_dict(nlp.make_doc(text), annots)
                                    for text, annots in train_data])

best_f1 = 0.0
history = []

for epoch in range(N_EPOCHS):
    random.shuffle(train_data)
    losses = {}

    batches = minibatch(train_data, size=4)
    for batch in batches:
        examples = [spacy.training.Example.from_dict(nlp.make_doc(text), annots) for text, annots in batch]
        nlp.update(examples, drop=0.2, sgd=optimizer, losses=losses)  # ✅ use sgd=optimizer

    # ===== Validation =====
    val_examples = [spacy.training.Example.from_dict(nlp.make_doc(text), annots) for text, annots in val_data]
    scorer = nlp.evaluate(val_examples)
    
    # Overall metrics
    f1 = scorer["ents_f"]
    p = scorer["ents_p"]
    r = scorer["ents_r"]
    
    print(f"Epoch {epoch+1}/{N_EPOCHS} | Loss: {losses.get('ner', 0):.3f} | Val F1={f1:.3f} | P={p:.3f} | R={r:.3f}")
    
    # ===== Per-label metrics =====
    per_label_metrics = scorer.get("ents_per_type", {})
    for label, metrics in per_label_metrics.items():
        print(f"  {label}: P={metrics['p']:.3f}, R={metrics['r']:.3f}, F1={metrics['f']:.3f}")
    
    # ===== Logging =====
    epoch_log = {
        "epoch": epoch + 1,
        "overall_f1": f1,
        "overall_precision": p,
        "overall_recall": r,
        "loss": losses.get("ner", 0),
    }
    
    # Flatten per-label metrics for saving (like your previous code)
    for label, metrics in per_label_metrics.items():
        epoch_log[f"precision_{label}"] = metrics["p"]
        epoch_log[f"recall_{label}"] = metrics["r"]
        epoch_log[f"f1_{label}"] = metrics["f"]
    
    history.append(epoch_log)

# ========= SAVE FINAL =========
OUTPUT_DIR.mkdir(exist_ok=True)
nlp.to_disk(OUTPUT_DIR / "final_model")
with open(OUTPUT_DIR / "training_history.json", "w") as f:
    json.dump(history, f, indent=2)

print("🎯 Training complete!")

[2025-10-07 19:46:05,529] [INFO] Created vocabulary
[2025-10-07 19:46:05,530] [INFO] Finished initializing nlp object


🟢 Using GPU


TypeError: Argument 'x' has incorrect type (expected numpy.ndarray, got ndarray)